In [1]:
import pandas as pd

df = pd.read_csv("../data/kaggle/train.csv")
print(df.shape)
df.head()

(94477, 4)


,path,participant_id,sequence_id,sign
0,train_landmark_files/26734/1000035562.parquet,26734,1000035562,blow
1,train_landmark_files/28656/1000106739.parquet,28656,1000106739,wait
2,train_landmark_files/16069/100015657.parquet,16069,100015657,cloud
3,train_landmark_files/25571/1000210073.parquet,25571,1000210073,bird
4,train_landmark_files/62590/1000240708.parquet,62590,1000240708,owie


In [2]:
print("Sequences:", len(df))
print("Participants:", df["participant_id"].nunique())
print("Signs:", df["sign"].nunique())

Sequences: 94477
Participants: 21
Signs: 250


In [3]:
counts = df["participant_id"].value_counts()
print(counts)
print("Min:", counts.min(), " Max:", counts.max())

participant_id
49445    4968
61333    4900
36257    4896
16069    4848
26734    4841
55372    4826
2044     4810
37779    4782
32319    4753
29302    4722
22343    4677
53618    4656
37055    4648
28656    4563
62590    4563
34503    4545
27610    4275
25571    3865
18796    3502
4718     3499
30680    3338
Name: count, dtype: int64
Min: 3338  Max: 4968


In [4]:
sample = df[df["sign"] == "cloud"].drop_duplicates("participant_id").head(3)
sample

,path,participant_id,sequence_id,sign
2,train_landmark_files/16069/100015657.parquet,16069,100015657,cloud
45,train_landmark_files/26734/1001931356.parquet,26734,1001931356,cloud
502,train_landmark_files/2044/1021210853.parquet,2044,1021210853,cloud


In [5]:
sample["path"].to_csv("../data/kaggle/sample_paths.txt", index=False, header=False, lineterminator="\n")

In [7]:
pq = pd.read_parquet("../data/kaggle/" + sample["path"].iloc[0])
print(pq.shape)
pq.head(10)

(57015, 7)


,frame,row_id,type,landmark_index,x,y,z
0,103,103-face-0,face,0,0.437886,0.437599,-0.051134
1,103,103-face-1,face,1,0.443258,0.392901,-0.067054
2,103,103-face-2,face,2,0.443997,0.409998,-0.042990
3,103,103-face-3,face,3,0.435256,0.362771,-0.039492
4,103,103-face-4,face,4,0.443780,0.381762,-0.068013
5,103,103-face-5,face,5,0.445125,0.369837,-0.058705
6,103,103-face-6,face,6,0.448380,0.342522,-0.012701
7,103,103-face-7,face,7,0.358520,0.352592,0.034469
8,103,103-face-8,face,8,0.449101,0.312440,0.005662
9,103,103-face-9,face,9,0.449679,0.296416,0.009398


In [8]:
print(pq["type"].value_counts())
print(pq.groupby("type")["landmark_index"].nunique())
print("Frames:", pq["frame"].nunique())

type
face          49140
pose           3465
left_hand      2205
right_hand     2205
Name: count, dtype: int64
type
face          468
left_hand      21
pose           33
right_hand     21
Name: landmark_index, dtype: int64
Frames: 105


In [9]:
print(pq.isna().sum())
print(pq[pq["x"].isna()].groupby("type")["frame"].nunique())

frame                0
row_id               0
type                 0
landmark_index       0
x                 3822
y                 3822
z                 3822
dtype: int64
type
left_hand      77
right_hand    105
Name: frame, dtype: int64


In [10]:
for path in sample["path"]:
    f = pd.read_parquet("../data/kaggle/" + path)
    n = f["frame"].nunique()
    lh_missing = f[(f["type"] == "left_hand") & f["x"].isna()]["frame"].nunique()
    rh_missing = f[(f["type"] == "right_hand") & f["x"].isna()]["frame"].nunique()
    print(path, "frames:", n, "| left missing:", lh_missing, "| right missing:", rh_missing)
    print("   x:", f["x"].min(), f["x"].max(), " y:", f["y"].min(), f["y"].max(), " z:", f["z"].min(), f["z"].max())

train_landmark_files/16069/100015657.parquet frames: 105 | left missing: 77 | right missing: 105
   x: -0.04292302206158638 1.197835922241211  y: 0.24809427559375763 2.591289520263672  z: -2.838325262069702 1.5875025987625122
train_landmark_files/26734/1001931356.parquet frames: 225 | left missing: 225 | right missing: 79
   x: -0.25927188992500305 1.4030174016952515  y: 0.08083554357290268 2.215975522994995  z: -2.9108588695526123 2.328218936920166
train_landmark_files/2044/1021210853.parquet frames: 16 | left missing: 16 | right missing: 8
   x: -0.2684774100780487 1.0652542114257812  y: 0.30798399448394775 2.523672103881836  z: -3.6682991981506348 2.3858020305633545
